# — Analyse d'un championnat de football avec Pandas


**LICENCE 3 - INTELLIGENCE ARTIFICIELLE** · Python Avancé · ISM École d'ingénieurs · Enseignant : Elias BADJI

Analyse de la Ligue Pro (championnat fictif de 20 clubs) sur trois saisons : 2023-2024, 2024-2025 et 2025-2026.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# Partie 1 — Chargement et diagnostic

**1.1** Chargement des quatre fichiers. La colonne `date` de `matchs` est lue comme une date avec `parse_dates`.

In [2]:
equipes = pd.read_csv("data/equipes.csv")
matchs = pd.read_csv("data/matchs.csv", parse_dates=["date"])
joueurs = pd.read_csv("data/joueurs.csv")
buts = pd.read_csv("data/buts.csv")

**1.2** Forme et types de chaque DataFrame.

In [3]:
tables = {"equipes": equipes, "matchs": matchs, "joueurs": joueurs, "buts": buts}
for nom, df in tables.items():
    print(f"===== {nom} : {df.shape[0]} lignes x {df.shape[1]} colonnes =====")
    df.info()
    print()

===== equipes : 20 lignes x 8 colonnes =====
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   equipe_id             20 non-null     str  
 1   nom                   20 non-null     str  
 2   ville                 20 non-null     str  
 3   region                20 non-null     str  
 4   annee_creation        20 non-null     int64
 5   stade                 20 non-null     str  
 6   capacite_stade        20 non-null     int64
 7   budget_millions_fcfa  20 non-null     int64
dtypes: int64(3), str(5)
memory usage: 1.4 KB

===== matchs : 1145 lignes x 13 colonnes =====
<class 'pandas.DataFrame'>
RangeIndex: 1145 entries, 0 to 1144
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   match_id            1145 non-null   str           
 1   saison     

**Interprétation 1.2 :** les formes correspondent à l'énoncé (20, 1 145, 1 688 et 2 660 lignes). `matchs.date` est bien en `datetime64`. En revanche `joueurs.salaire_mensuel_fcfa` est de type texte (`object`/`str`) alors qu'elle devrait être numérique, et `buts.minute` est du texte à cause des temps additionnels.

**1.3** Type de `salaire_mensuel_fcfa` et valeurs qui empêchent la conversion.

In [4]:
sal = joueurs["salaire_mensuel_fcfa"]
print("Type :", sal.dtype)
non_convertibles = sal[pd.to_numeric(sal, errors="coerce").isna() & sal.notna()]
print("Nombre de valeurs non convertibles :", len(non_convertibles))
print(non_convertibles.drop_duplicates().head(10).tolist())

Type : str
Nombre de valeurs non convertibles : 40
['450 000', '510 000', '350 000', '430 000', '390 000', '210 000', '260 000', '470 000', '670 000', '530 000']


**Interprétation 1.3 :** la colonne est de type texte. Certains salaires sont saisis avec un espace comme séparateur de milliers (par exemple `'450 000'`), ce que `pd.to_numeric` ne sait pas lire : il faudra retirer les espaces avant la conversion.

**1.4** Valeurs manquantes par colonne, dans les quatre tables.

In [5]:
for nom, df in tables.items():
    print(f"--- {nom} ---")
    manquants = df.isna().sum()
    print(manquants[manquants > 0] if manquants.sum() > 0 else "Aucune valeur manquante")
    print()

--- equipes ---
Aucune valeur manquante

--- matchs ---
spectateurs    18
dtype: int64

--- joueurs ---
date_naissance          10
salaire_mensuel_fcfa    15
minutes_jouees           8
dtype: int64

--- buts ---
passeur_id    935
dtype: int64



In [6]:
# Les passeur_id manquants : selon le type de but ?
print(buts.groupby("type_but")["passeur_id"].apply(lambda s: s.isna().sum()).rename("passeurs_manquants"))
print("Penalties :", (buts["type_but"] == "penalty").sum())

type_but
jeu        652
penalty    283
Name: passeurs_manquants, dtype: int64
Penalties : 283


**Interprétation 1.4 :** manquants : `spectateurs` (matchs), `date_naissance`, `salaire_mensuel_fcfa` et `minutes_jouees` (joueurs), `passeur_id` (buts). Les `passeur_id` manquants **ne sont pas une erreur** : un but sans passe décisive (action individuelle, penalty) n'a légitimement pas de passeur. Tous les penalties n'ont pas de passeur, ce qui est cohérent. Il ne faut donc surtout pas les imputer.

**1.5** Lignes en double et clés en double.

In [7]:
for nom, df in tables.items():
    print(f"{nom:8s} lignes dupliquées : {df.duplicated().sum()}")
print()
print("match_id dupliqués (matchs) :", matchs["match_id"].duplicated().sum())
print("but_id dupliqués (buts)     :", buts["but_id"].duplicated().sum())
print("(saison, joueur_id) dupliqués (joueurs) :", joueurs.duplicated(["saison", "joueur_id"]).sum())

equipes  lignes dupliquées : 0
matchs   lignes dupliquées : 5
joueurs  lignes dupliquées : 8
buts     lignes dupliquées : 6

match_id dupliqués (matchs) : 5
but_id dupliqués (buts)     : 6
(saison, joueur_id) dupliqués (joueurs) : 8


**Interprétation 1.5 :** il y a 5 matchs, 8 lignes joueur-saison et 6 buts en doublon exact (même nombre de lignes dupliquées que de clés dupliquées : ce sont de vraies copies, pas des conflits de valeurs). Supprimer les doublons suffira.

**1.6** Valeurs distinctes de `poste`.

In [8]:
print(joueurs["poste"].value_counts())
print("\nNombre de variantes :", joueurs["poste"].nunique(), "pour 4 postes")

poste
Défenseur     536
Milieu        534
Attaquant     415
Gardien       179
défenseur       2
Attaquant       2
gardien         2
milieu          2
MILIEU          2
Milieu          2
Defenseur       2
DEFENSEUR       2
Défenseur       2
attaquant       2
ATTAQUANT       2
 Milieu         2
Name: count, dtype: int64

Nombre de variantes : 16 pour 4 postes


**Interprétation 1.6 :** on trouve 16 variantes pour 4 postes, à cause de la casse (`MILIEU`, `milieu`), des espaces autour (`' Milieu '`) et des accents oubliés (`Defenseur`).

**1.7** Buts avec une minute du type « 90+3 ».

In [9]:
nb_add = buts["minute"].str.contains(r"\+").sum()
print("Buts en temps additionnel :", nb_add, f"({nb_add / len(buts):.1%})")
buts.loc[buts["minute"].str.contains(r"\+"), "minute"].value_counts().sort_index()

Buts en temps additionnel : 228 (8.6%)


minute
45+1    16
45+2    12
45+3    15
90+1    40
90+2    40
90+3    33
90+4    35
90+5    37
Name: count, dtype: int64

**Interprétation 1.7 :** 228 buts (8,6 %) ont une minute au format « 45+x » ou « 90+x » ; ces valeurs obligent à garder `minute` en texte, d'où la nécessité de la décomposer en deux colonnes numériques.

**1.8 Problèmes trouvés :**

- `joueurs.salaire_mensuel_fcfa` est stockée en texte, avec des espaces comme séparateurs de milliers.
- `joueurs.salaire_mensuel_fcfa` : 15 valeurs manquantes.
- `joueurs.poste` : 16 variantes (casse, espaces, accents) pour 4 postes.
- `joueurs.date_naissance` : 10 valeurs manquantes, et colonne lue comme texte.
- `joueurs.minutes_jouees` : 8 valeurs manquantes.
- `joueurs` : 8 lignes en double (clé (`saison`, `joueur_id`) non unique).
- `matchs` : 5 lignes en double (`match_id` non unique).
- `matchs.spectateurs` : 18 valeurs manquantes.
- `buts` : 6 lignes en double (`but_id` non unique).
- `buts.minute` : texte, dont 228 minutes avec temps additionnel (« 45+2 », « 90+3 »).
- `buts.passeur_id` : 935 manquants, mais **légitimes** (but sans passeur, penalties) : à conserver tels quels.

# Partie 2 — Nettoyage

Toutes les transformations se font sur des **copies** (`*_c`) afin de pouvoir comparer avant/après à la question 4.8.

**2.1** Suppression des doublons exacts : ce sont de simples copies d'une même saisie, on garde la première occurrence.

In [10]:
equipes_c = equipes.copy()
matchs_c = matchs.drop_duplicates().copy()
joueurs_c = joueurs.drop_duplicates().copy()
buts_c = buts.drop_duplicates().copy()

assert matchs_c["match_id"].is_unique
assert buts_c["but_id"].is_unique
assert not joueurs_c.duplicated(["saison", "joueur_id"]).any()
print("Unicité des clés vérifiée :", len(matchs_c), "matchs,", len(joueurs_c), "joueurs-saisons,", len(buts_c), "buts")

Unicité des clés vérifiée : 1140 matchs, 1680 joueurs-saisons, 2654 buts


**Interprétation 2.1 :** après suppression, `match_id`, `but_id` et (`saison`, `joueur_id`) sont tous uniques : les doublons étaient bien de pures répétitions.

**2.2** Harmonisation de `poste` : on retire les espaces, on normalise la casse, puis on corrige l'accent manquant (`Defenseur` → `Défenseur`).

In [11]:
joueurs_c["poste"] = (
    joueurs_c["poste"]
    .str.strip()
    .str.capitalize()
    .replace({"Defenseur": "Défenseur"})
)
joueurs_c["poste"].value_counts()

poste
Défenseur    540
Milieu       540
Attaquant    420
Gardien      180
Name: count, dtype: int64

**Interprétation 2.2 :** il ne reste que les quatre postes officiels ; les regroupements par poste (question 2.4) seront maintenant fiables.

**2.3** Conversion de `salaire_mensuel_fcfa` : on supprime les espaces (séparateurs de milliers) puis on convertit en nombre ; les manquants restent `NaN`.

In [12]:
joueurs_c["salaire_mensuel_fcfa"] = pd.to_numeric(
    joueurs_c["salaire_mensuel_fcfa"].str.replace(r"\s+", "", regex=True)
)
print(joueurs_c["salaire_mensuel_fcfa"].dtype, "| manquants :", joueurs_c["salaire_mensuel_fcfa"].isna().sum())

float64 | manquants : 15


**Interprétation 2.3 :** la colonne est désormais numérique (`float64`) et aucune valeur n'a été perdue lors de la conversion : seuls les vrais manquants restent à traiter.

**2.4** Imputation des salaires manquants par la médiane du **même poste et de la même saison**.

In [13]:
mediane_groupe = joueurs_c.groupby(["saison", "poste"])["salaire_mensuel_fcfa"].transform("median")
joueurs_c["salaire_mensuel_fcfa"] = joueurs_c["salaire_mensuel_fcfa"].fillna(mediane_groupe)
print("Salaires manquants restants :", joueurs_c["salaire_mensuel_fcfa"].isna().sum())
joueurs_c.groupby(["saison", "poste"])["salaire_mensuel_fcfa"].median().unstack()

Salaires manquants restants : 0


poste,Attaquant,Défenseur,Gardien,Milieu
saison,,,,
2023-2024,450000.0,330000.0,290000.0,370000.0
2024-2025,490000.0,360000.0,290000.0,390000.0
2025-2026,490000.0,370000.0,330000.0,420000.0


**Pourquoi ces deux critères ?** Le salaire dépend fortement du poste (un attaquant n'est pas payé comme un gardien) et évolue d'une saison à l'autre ; la médiane globale attribuerait à un joueur un salaire non représentatif de son profil. La médiane (plutôt que la moyenne) est de plus robuste aux très gros salaires.

**Interprétation 2.4 :** les 15 salaires manquants sont remplacés par une valeur plausible propre à leur poste et à leur saison, sans déformer la distribution.

**2.5** Conversion de `date_naissance` en date.

In [14]:
joueurs_c["date_naissance"] = pd.to_datetime(joueurs_c["date_naissance"], errors="coerce")
print("Manquants avant récupération :", joueurs_c["date_naissance"].isna().sum())

# Un joueur garde le même joueur_id d'une saison à l'autre : on peut récupérer sa date dans une autre saison
joueurs_c["date_naissance"] = joueurs_c["date_naissance"].fillna(
    joueurs_c.groupby("joueur_id")["date_naissance"].transform("first")
)
print("Manquants après récupération :", joueurs_c["date_naissance"].isna().sum())

Manquants avant récupération : 10
Manquants après récupération : 2


**Choix 2.5 :** on **n'invente pas** de date de naissance (une date « moyenne » serait fausse pour chaque joueur et fausserait les âges). En revanche, comme `joueur_id` est stable d'une saison à l'autre, on récupère la vraie date quand elle est renseignée sur une autre saison du même joueur. Les dates restantes sont laissées en `NaT` : cette colonne n'est utilisée dans aucune analyse du TP.

**Interprétation 2.5 :** une partie des dates manquantes est retrouvée sans aucune hypothèse ; le reste est laissé vide, ce qui est plus honnête qu'une valeur fabriquée.

**2.6** Décomposition de `minute` en `minute_num` (minute réglementaire) et `temps_add` (minutes additionnelles).

In [15]:
parts = buts_c["minute"].astype(str).str.split("+", expand=True)
buts_c["minute_num"] = parts[0].astype(int)
buts_c["temps_add"] = pd.to_numeric(parts[1]).fillna(0).astype(int)
buts_c[buts_c["temps_add"] > 0][["minute", "minute_num", "temps_add"]].head()

,minute,minute_num,temps_add
4,90+5,90,5
33,90+2,90,2
34,90+2,90,2
39,45+1,45,1
64,90+4,90,4


In [16]:
print(buts_c[["minute_num", "temps_add"]].describe().loc[["min", "max", "mean"]])

      minute_num  temps_add
min     1.000000   0.000000
max    90.000000   5.000000
mean   48.963451   0.235494


**Interprétation 2.6 :** « 45+2 » devient `minute_num = 45` et `temps_add = 2` ; les buts normaux ont `temps_add = 0`. On peut maintenant faire des calculs et des regroupements par tranche de temps.

**2.7** Spectateurs manquants dans `matchs`.

In [17]:
print("Manquants :", matchs_c["spectateurs"].isna().sum(), "sur", len(matchs_c))
matchs_c["spectateurs_impute"] = matchs_c["spectateurs"].isna()
mediane_club = matchs_c.groupby(["saison", "equipe_dom_id"])["spectateurs"].transform("median")
matchs_c["spectateurs"] = matchs_c["spectateurs"].fillna(mediane_club)
print("Manquants restants :", matchs_c["spectateurs"].isna().sum())

Manquants : 18 sur 1140
Manquants restants : 0


**Choix 2.7 :** l'affluence dépend avant tout du club qui reçoit (taille du stade, popularité) : on impute par la **médiane des matchs à domicile du même club dans la même saison**, et on conserve un indicateur `spectateurs_impute` pour garder la trace des valeurs reconstituées. Avec seulement 18 manquants sur 1 140 (1,6 %), l'impact sur les moyennes est négligeable. Supprimer les lignes aurait fait perdre des matchs entiers (et fausserait les classements).

**Interprétation 2.7 :** plus aucune affluence manquante, et les valeurs imputées restent réalistes pour le stade concerné.

**2.8** Contrôle final.

In [18]:
assert len(matchs_c) == 1140
assert len(joueurs_c) == 1680
assert len(buts_c) == 2654
print("OK :", len(matchs_c), "matchs,", len(joueurs_c), "lignes joueur-saison,", len(buts_c), "buts")

OK : 1140 matchs, 1680 lignes joueur-saison, 2654 buts


**Interprétation 2.8 :** les trois effectifs attendus (1 140 matchs, 1 680 lignes joueur-saison, 2 654 buts) sont obtenus : le nettoyage est cohérent avec l'énoncé.

# Partie 3 — Classements des trois saisons

**3.1** DataFrame `dom` : chaque match vu du côté du club qui reçoit.

In [19]:
dom = matchs_c.rename(columns={
    "equipe_dom_id": "equipe_id",
    "buts_dom": "buts_pour",
    "buts_ext": "buts_contre",
})[["match_id", "saison", "journee", "equipe_id", "buts_pour", "buts_contre"]]
dom.head()

,match_id,saison,journee,equipe_id,buts_pour,buts_contre
0,M0001,2023-2024,1,E01,1,2
1,M0002,2023-2024,1,E02,2,1
2,M0003,2023-2024,1,E03,1,1
3,M0004,2023-2024,1,E04,5,1
4,M0005,2023-2024,1,E05,3,2


**3.2** DataFrame `ext` : chaque match vu du côté du club qui se déplace (ses buts pour sont `buts_ext`, ses buts contre `buts_dom`).

In [20]:
ext = matchs_c.rename(columns={
    "equipe_ext_id": "equipe_id",
    "buts_ext": "buts_pour",
    "buts_dom": "buts_contre",
})[["match_id", "saison", "journee", "equipe_id", "buts_pour", "buts_contre"]]
ext.head()

,match_id,saison,journee,equipe_id,buts_pour,buts_contre
0,M0001,2023-2024,1,E20,2,1
1,M0002,2023-2024,1,E19,1,2
2,M0003,2023-2024,1,E18,1,1
3,M0004,2023-2024,1,E17,1,5
4,M0005,2023-2024,1,E16,2,3


**3.3** Empilement avec `pd.concat`. Attendu : 1 140 matchs × 2 = **2 280 lignes**.

In [21]:
resultats = pd.concat([dom, ext], ignore_index=True)
print(resultats.shape)
assert len(resultats) == 2 * len(matchs_c) == 2280

(2280, 6)


**Interprétation 3.3 :** `resultats` contient bien 2 280 lignes, soit deux par match (une par équipe).

**3.4** Colonnes `points`, `victoire`, `nul`, `defaite`.

In [22]:
resultats["points"] = np.select(
    [resultats["buts_pour"] > resultats["buts_contre"], resultats["buts_pour"] == resultats["buts_contre"]],
    [3, 1],
    default=0,
)
resultats["victoire"] = (resultats["buts_pour"] > resultats["buts_contre"]).astype(int)
resultats["nul"] = (resultats["buts_pour"] == resultats["buts_contre"]).astype(int)
resultats["defaite"] = (resultats["buts_pour"] < resultats["buts_contre"]).astype(int)
resultats.head()

,match_id,saison,journee,equipe_id,buts_pour,buts_contre,points,victoire,nul,defaite
0,M0001,2023-2024,1,E01,1,2,0,0,0,1
1,M0002,2023-2024,1,E02,2,1,3,1,0,0
2,M0003,2023-2024,1,E03,1,1,1,0,1,0
3,M0004,2023-2024,1,E04,5,1,3,1,0,0
4,M0005,2023-2024,1,E05,3,2,3,1,0,0


**Interprétation 3.4 :** chaque ligne de `resultats` porte maintenant ses points (3, 1 ou 0) et l'indicateur de victoire, nul ou défaite correspondant.

**3.5** Agrégation nommée par (`saison`, `equipe_id`).

In [23]:
classement = (
    resultats.groupby(["saison", "equipe_id"])
    .agg(
        J=("match_id", "count"),
        G=("victoire", "sum"),
        N=("nul", "sum"),
        P=("defaite", "sum"),
        BP=("buts_pour", "sum"),
        BC=("buts_contre", "sum"),
        Pts=("points", "sum"),
    )
    .reset_index()
)
classement["Diff"] = classement["BP"] - classement["BC"]
classement.head()

,saison,equipe_id,J,G,N,P,BP,BC,Pts,Diff
0,2023-2024,E01,38,27,7,4,71,33,88,38
1,2023-2024,E02,38,14,10,14,51,43,52,8
2,2023-2024,E03,38,22,6,10,54,32,72,22
3,2023-2024,E04,38,17,11,10,53,46,62,7
4,2023-2024,E05,38,13,13,12,44,44,52,0


**Interprétation 3.5 :** chaque club a 38 matchs (`J`) par saison, et `Pts = 3×G + N` ; la différence de buts mesure l'équilibre attaque-défense.

**3.6** Ajout du nom des clubs, tri selon le règlement (saison ↑, puis Pts, Diff et BP ↓) et colonne `rang`.

In [24]:
classement = classement.merge(equipes_c[["equipe_id", "nom"]], on="equipe_id", how="left")
classement = classement.sort_values(
    ["saison", "Pts", "Diff", "BP"], ascending=[True, False, False, False]
).reset_index(drop=True)
classement["rang"] = classement.groupby("saison").cumcount() + 1
classement = classement[["saison", "rang", "equipe_id", "nom", "J", "G", "N", "P", "BP", "BC", "Diff", "Pts"]]
classement.head(5)

,saison,rang,equipe_id,nom,J,G,N,P,BP,BC,Diff,Pts
0,2023-2024,1,E01,Étoile du Cap-Vert,38,27,7,4,71,33,38,88
1,2023-2024,2,E03,Fleuve Athletic,38,22,6,10,54,32,22,72
2,2023-2024,3,E17,Diourbel Sporting,38,21,7,10,49,39,10,70
3,2023-2024,4,E20,Banlieue Espoir,38,18,9,11,59,33,26,63
4,2023-2024,5,E04,Palmiers FC,38,17,11,10,53,46,7,62


**Interprétation 3.6 :** le classement est trié selon le règlement et chaque club reçoit son `rang` de 1 à 20 dans sa saison.

**3.7** Contrôles de cohérence.

In [25]:
# 38 matchs par club et par saison
assert (classement["J"] == 38).all()
# Dans chaque saison : buts pour = buts contre
par_saison = classement.groupby("saison")[["BP", "BC", "G", "P", "N"]].sum()
assert (par_saison["BP"] == par_saison["BC"]).all()
# Dans chaque saison : victoires = défaites
assert (par_saison["G"] == par_saison["P"]).all()
print("Tous les contrôles sont passés.")
par_saison

Tous les contrôles sont passés.


,BP,BC,G,P,N
saison,,,,,
2023-2024,949,949,285,285,190
2024-2025,843,843,294,294,172
2025-2026,862,862,264,264,232


**Interprétation 3.7 :** tous les `assert` passent : chaque club a joué 38 matchs, et dans chaque saison les buts marqués égalent les buts encaissés et les victoires égalent les défaites (chaque victoire correspond à une défaite adverse). La transformation en deux lignes par match est donc correcte.

**3.8** Podium de chaque saison.

In [26]:
podium = classement[classement["rang"] <= 3]
print(podium[["saison", "rang", "nom", "Pts", "Diff"]].to_string(index=False))
print()
presences = podium.groupby("nom")["saison"].nunique().sort_values(ascending=False)
print("Clubs sur le podium :")
print(presences)

   saison  rang                nom  Pts  Diff
2023-2024     1 Étoile du Cap-Vert   88    38
2023-2024     2    Fleuve Athletic   72    22
2023-2024     3  Diourbel Sporting   70    10
2024-2025     1        Palmiers FC   79    32
2024-2025     2 Étoile du Cap-Vert   75    37
2024-2025     3    Fleuve Athletic   71    28
2025-2026     1  Presqu'île Racing   75    32
2025-2026     2 Étoile du Cap-Vert   73    36
2025-2026     3    Baobab Sporting   67    29

Clubs sur le podium :
nom
Étoile du Cap-Vert    3
Fleuve Athletic       2
Diourbel Sporting     1
Baobab Sporting       1
Palmiers FC           1
Presqu'île Racing     1
Name: saison, dtype: int64


**Interprétation 3.8 :** oui, **Étoile du Cap-Vert** est sur le podium les trois saisons (1er en 2023-2024, 2e en 2024-2025 et en 2025-2026). Les champions sont Étoile du Cap-Vert (88 pts), Palmiers FC (79 pts) puis Presqu'île Racing (75 pts) : le titre change de mains chaque saison et l'écart du champion se resserre.

**3.9** Classement complet de 2025-2026 et clubs à égalité parfaite (points et différence de buts).

In [27]:
c2526 = classement[classement["saison"] == "2025-2026"]
print(c2526.drop(columns="saison").to_string(index=False))
print()
egalite = c2526[c2526.duplicated(["Pts", "Diff"], keep=False)]
print("Clubs à égalité de points ET de différence de buts :")
print(egalite[["rang", "nom", "Pts", "Diff", "BP"]].to_string(index=False))

 rang equipe_id                nom  J  G  N  P  BP  BC  Diff  Pts
    1       E09  Presqu'île Racing 38 22  9  7  58  26    32   75
    2       E01 Étoile du Cap-Vert 38 21 10  7  76  40    36   73
    3       E02    Baobab Sporting 38 17 16  5  60  31    29   67
    4       E20    Banlieue Espoir 38 19  9 10  51  41    10   66
    5       E04        Palmiers FC 38 19  7 12  54  41    13   64
    6       E10    Niayes Académie 38 17  8 13  46  40     6   59
    7       E13        Sine Racing 38 15 11 12  42  41     1   56
    8       E05      Saloum United 38 12 17  9  44  36     8   53
    9       E15         Delta Club 38 13 12 13  29  45   -16   51
   10       E06         Océan Club 38 13 11 14  42  48    -6   50
   11       E03    Fleuve Athletic 38 12 12 14  48  45     3   48
   12       E14       Ferlo United 38 12 11 15  38  43    -5   47
   13       E16           Forêt FC 38 11 14 13  36  41    -5   47
   14       E17  Diourbel Sporting 38 11 12 15  35  48   -13   45
   15     

**Interprétation 3.9 :** **Ferlo United** et **Forêt FC** terminent tous deux à 47 points avec une différence de buts de −5. Ferlo United est devant (12e) car il a marqué plus de buts (38 contre 36), troisième critère du règlement.

# Partie 4 — Analyses pour le rapport

**4.1 Meilleurs buteurs 2025-2026.**

In [28]:
clubs = equipes_c[["equipe_id", "nom"]].rename(columns={"nom": "club"})
buteurs = (
    joueurs_c[joueurs_c["saison"] == "2025-2026"]
    .merge(clubs, on="equipe_id", how="left")
    .sort_values(["buts", "passes_decisives"], ascending=False)
    .head(5)[["prenom", "nom", "club", "buts", "passes_decisives"]]
    .reset_index(drop=True)
)
buteurs

,prenom,nom,club,buts,passes_decisives
0,Modou,Thiam,Niayes Académie,19,2
1,Bakary,Wade,Étoile du Cap-Vert,17,5
2,Bamba,Badji,Océan Club,15,5
3,Amadou,Kâ,Banlieue Espoir,15,1
4,Omar,Cissé,Saloum United,15,0


**Interprétation 4.1 :** **Modou Thiam** (Niayes Académie) est le meilleur buteur 2025-2026 avec 19 buts, devant Bakary Wade (17, Étoile du Cap-Vert). Trois joueurs sont à 15 buts pour les 3e-5e places ; ils sont ici départagés par les passes décisives, et d'autres joueurs suivent à 14 buts.#%% md
**4.2 Avantage du terrain.**

In [29]:
matchs_c["resultat"] = np.select(
    [matchs_c["buts_dom"] > matchs_c["buts_ext"], matchs_c["buts_dom"] == matchs_c["buts_ext"]],
    ["Victoire domicile", "Nul"],
    default="Victoire extérieur",
)
avantage = pd.crosstab(matchs_c["saison"], matchs_c["resultat"], normalize="index").mul(100).round(1)
avantage[["Victoire domicile", "Nul", "Victoire extérieur"]]

resultat,Victoire domicile,Nul,Victoire extérieur
saison,,,
2023-2024,48.2,25.0,26.8
2024-2025,48.4,22.6,28.9
2025-2026,43.2,30.5,26.3


**Interprétation 4.2 :** la victoire à domicile reste le résultat le plus fréquent chaque saison (48,2 %, 48,4 % puis 43,2 %), mais l'avantage n'est pas parfaitement stable : en 2025-2026 il recule d'environ 5 points au profit des matchs nuls (30,5 % contre 22,6 % l'année précédente), tandis que les victoires à l'extérieur restent autour de 26-29 %.

**4.3 Affluence 2025-2026** : affluence moyenne à domicile et taux de remplissage du stade.

In [30]:
affluence = (
    matchs_c[matchs_c["saison"] == "2025-2026"]
    .groupby("equipe_dom_id")["spectateurs"].mean()
    .rename("affluence_moyenne")
    .reset_index()
    .merge(equipes_c[["equipe_id", "nom", "stade", "capacite_stade"]], left_on="equipe_dom_id", right_on="equipe_id")
)
affluence["taux_remplissage_pct"] = (affluence["affluence_moyenne"] / affluence["capacite_stade"] * 100).round(1)
affluence["affluence_moyenne"] = affluence["affluence_moyenne"].round(0)
affluence = affluence[["nom", "stade", "capacite_stade", "affluence_moyenne", "taux_remplissage_pct"]] \
    .sort_values("taux_remplissage_pct", ascending=False).reset_index(drop=True)
affluence

,nom,stade,capacite_stade,affluence_moyenne,taux_remplissage_pct
0,Étoile du Cap-Vert,Stade du Cap,18000,14559.0,80.9
1,Banlieue Espoir,Stade de la Banlieue,8000,6213.0,77.7
2,Presqu'île Racing,Stade de la Corniche,11000,8305.0,75.5
3,Forêt FC,Stade de la Forêt,6000,4415.0,73.6
4,Palmiers FC,Stade des Palmiers,10000,7123.0,71.2
5,Saloum United,Stade du Saloum,8000,5607.0,70.1
6,Niayes Académie,Stade des Niayes,4000,2760.0,69.0
7,Baobab Sporting,Stade des Baobabs,12000,8276.0,69.0
8,Sahel Olympique,Stade du Sahel,6000,4069.0,67.8
9,Fleuve Athletic,Stade du Fleuve,9000,6064.0,67.4


In [31]:
affluence.head(3)

,nom,stade,capacite_stade,affluence_moyenne,taux_remplissage_pct
0,Étoile du Cap-Vert,Stade du Cap,18000,14559.0,80.9
1,Banlieue Espoir,Stade de la Banlieue,8000,6213.0,77.7
2,Presqu'île Racing,Stade de la Corniche,11000,8305.0,75.5


**Interprétation 4.3 :** les trois clubs qui remplissent le mieux leur stade sont **Étoile du Cap-Vert (80,9 %)**, **Banlieue Espoir (77,7 %)** et **Presqu'île Racing (75,5 %)**. Aucun stade n'est plein en moyenne, et les clubs de tête figurent parmi les mieux remplis.

**4.4 Quand marque-t-on ?** Tranches de 15 minutes sur `minute_num`.

In [32]:
tranches = pd.cut(
    buts_c["minute_num"],
    bins=[0, 15, 30, 45, 60, 75, 90],
    labels=["1-15", "16-30", "31-45", "46-60", "61-75", "76-90"],
)
part_tranches = tranches.value_counts(normalize=True).sort_index().mul(100).round(1)
print(part_tranches.rename("part des buts (%)"))
part_add = (buts_c["temps_add"] > 0).mean() * 100
print(f"\nPart des buts marqués dans le temps additionnel : {part_add:.1f} %")
print(buts_c.loc[buts_c['temps_add'] > 0].groupby('minute_num').size().rename('buts en temps additionnel'))

minute_num
1-15     15.0
16-30    14.8
31-45    18.5
46-60    13.3
61-75    16.1
76-90    22.5
Name: part des buts (%), dtype: float64

Part des buts marqués dans le temps additionnel : 8.6 %
minute_num
45     43
90    184
Name: buts en temps additionnel, dtype: int64


**Interprétation 4.4 :** les buts sont plus fréquents en fin de match : la tranche 76-90 concentre 22,5 % des buts (la plus riche) et la tranche 46-60 seulement 13,3 % (la plus pauvre). Le temps additionnel pèse pour **8,6 %** de l'ensemble des buts (184 buts à la fin du 2e temps, 43 à la fin du 1er) : sans lui, la tranche 76-90 tomberait à environ 15,6 %, ce qui montre que le pic de fin de match vient surtout des arrêts de jeu.

**4.5 Penalties et passes.**

In [33]:
part_penalty = (buts_c["type_but"] == "penalty").mean() * 100
part_sans_passeur = buts_c["passeur_id"].isna().mean() * 100
hors_penalty = buts_c[buts_c["type_but"] == "jeu"]
part_sans_passeur_jeu = hors_penalty["passeur_id"].isna().mean() * 100
print(f"Part des buts sur penalty : {part_penalty:.1f} %")
print(f"Part des buts sans passeur : {part_sans_passeur:.1f} %")
print(f"Part des buts de jeu (hors penalty) sans passeur : {part_sans_passeur_jeu:.1f} %")

Part des buts sur penalty : 10.6 %
Part des buts sans passeur : 35.2 %
Part des buts de jeu (hors penalty) sans passeur : 27.5 %


**Interprétation 4.5 (parts) :** environ 10,6 % des buts viennent d'un penalty. 35,2 % des buts n'ont pas de passeur ; comme un penalty n'en a jamais, seuls 27,5 % des buts de jeu sont des actions individuelles : près des trois quarts des buts de jeu sont donc assistés.

In [34]:
identite = joueurs_c[["joueur_id", "prenom", "nom"]].drop_duplicates("joueur_id")
identite["joueur"] = identite["prenom"] + " " + identite["nom"]

duos = (
    buts_c.dropna(subset=["passeur_id"])
    .groupby(["passeur_id", "buteur_id"]).size()
    .rename("nb_buts").reset_index()
    .nlargest(3, "nb_buts", keep="all")   # keep="all" : on garde tous les duos à égalité sur la 3e place
)
duos = (
    duos
    .merge(identite[["joueur_id", "joueur"]].rename(columns={"joueur_id": "passeur_id", "joueur": "passeur"}), on="passeur_id")
    .merge(identite[["joueur_id", "joueur"]].rename(columns={"joueur_id": "buteur_id", "joueur": "buteur"}), on="buteur_id")
    .sort_values("nb_buts", ascending=False)
)
duos[["passeur", "buteur", "nb_buts"]].reset_index(drop=True)

,passeur,buteur,nb_buts
0,Malick Diatta,Khadim Diop,6
1,Thierno Wade,Bakary Mendy,5
2,Mouhamed Coly,Modou Thiam,5
3,Youssou Diallo,Amadou Kâ,5
4,Ousmane Faye,Modou Thiam,5


**Interprétation 4.5 (duos) :** le duo le plus prolifique est **Malick Diatta → Khadim Diop** avec 6 buts. La 2e place est partagée entre quatre duos à 5 buts (dont Mouhamed Coly → Modou Thiam et Ousmane Faye → Modou Thiam, qui montrent que le meilleur buteur profite de plusieurs passeurs) : `keep="all"` conserve toutes les égalités plutôt que d'en choisir arbitrairement.

**4.6 Argent et résultats (2025-2026).**

In [35]:
masse = (
    joueurs_c[joueurs_c["saison"] == "2025-2026"]
    .groupby("equipe_id")["salaire_mensuel_fcfa"].sum()
    .rename("masse_salariale_mensuelle")
    .reset_index()
)
argent = (
    classement[classement["saison"] == "2025-2026"]
    .merge(masse, on="equipe_id")
    [["rang", "nom", "Pts", "masse_salariale_mensuelle"]]
)
print(argent.to_string(index=False))
print()
print("Corrélation de Pearson  (masse salariale, points) :", round(argent["masse_salariale_mensuelle"].corr(argent["Pts"]), 3))
print("Corrélation de Spearman (masse salariale, points) :", round(argent["masse_salariale_mensuelle"].corr(argent["Pts"], method="spearman"), 3))

 rang                nom  Pts  masse_salariale_mensuelle
    1  Presqu'île Racing   75                 16830000.0
    2 Étoile du Cap-Vert   73                 18970000.0
    3    Baobab Sporting   67                 16010000.0
    4    Banlieue Espoir   66                 14580000.0
    5        Palmiers FC   64                 14430000.0
    6    Niayes Académie   59                 11750000.0
    7        Sine Racing   56                 10320000.0
    8      Saloum United   53                 12040000.0
    9         Delta Club   51                  9350000.0
   10         Océan Club   50                 13310000.0
   11    Fleuve Athletic   48                 14700000.0
   12       Ferlo United   47                  9150000.0
   13           Forêt FC   47                 10660000.0
   14  Diourbel Sporting   45                 12520000.0
   15    Lagune Sportive   42                  9670000.0
   16          Savane FC   40                 10040000.0
   17         Falaise AC   40  

**Interprétation 4.6 :** la corrélation entre masse salariale et points est forte et positive (Pearson 0,82 ; Spearman 0,75) : les clubs qui paient le plus ont tendance à finir devant. **On peut conclure** à une association solide entre argent et résultats cette saison. **On ne peut pas conclure** que l'argent *cause* les points : les clubs performants peuvent aussi gagner plus et donc payer plus (causalité inverse), la taille du club joue sur les deux, et il n'y a que 20 clubs sur une seule saison. Les exceptions le montrent : Étoile du Cap-Vert a la plus forte masse salariale mais finit 2e derrière Presqu'île Racing, et Océan Club (13,3 M FCFA) n'est que 10e.

**4.7 Carrières.**

In [36]:
carrieres = joueurs_c.groupby("joueur_id").agg(
    nb_saisons=("saison", "nunique"),
    nb_clubs=("equipe_id", "nunique"),
)
print("Joueurs différents sur trois saisons :", len(carrieres))
print("Joueurs ayant joué les trois saisons :", (carrieres["nb_saisons"] == 3).sum())
print("Joueurs ayant porté plusieurs maillots :", (carrieres["nb_clubs"] > 1).sum())
print()
print(carrieres["nb_saisons"].value_counts().sort_index().rename("joueurs par nombre de saisons"))

Joueurs différents sur trois saisons : 717
Joueurs ayant joué les trois saisons : 417
Joueurs ayant porté plusieurs maillots : 69

nb_saisons
1    171
2    129
3    417
Name: joueurs par nombre de saisons, dtype: int64


**Interprétation 4.7 :** la Ligue compte **717 joueurs différents** sur trois saisons, dont **417 (58 %)** ont joué les trois saisons : l'effectif est assez stable. **69 joueurs** ont porté le maillot de plusieurs clubs, soit moins de 10 % : les transferts restent minoritaires.

**4.8 Contrôles croisés.**

In [37]:
# (a) pour chaque match et chaque équipe : nb de lignes de buts == score
def controle_scores(df_buts):
    nb = df_buts.groupby(["match_id", "equipe_id"]).size().rename("nb_buts_table").reset_index()
    ctrl = resultats[["match_id", "equipe_id", "buts_pour"]].merge(nb, on=["match_id", "equipe_id"], how="left")
    ctrl["nb_buts_table"] = ctrl["nb_buts_table"].fillna(0).astype(int)   # 0-0 : aucune ligne dans buts
    return ctrl

ctrl_propre = controle_scores(buts_c)
ecarts_propre = (ctrl_propre["buts_pour"] != ctrl_propre["nb_buts_table"]).sum()
assert ecarts_propre == 0
print("Scores incohérents (tables nettoyées) :", ecarts_propre)

# Même contrôle avec le fichier buts.csv BRUT (avec doublons)
ctrl_brut = controle_scores(buts)
ecarts_brut = ctrl_brut[ctrl_brut["buts_pour"] != ctrl_brut["nb_buts_table"]]
print("Scores incohérents (buts.csv brut)   :", len(ecarts_brut))
ecarts_brut

Scores incohérents (tables nettoyées) : 0
Scores incohérents (buts.csv brut)   : 6


,match_id,equipe_id,buts_pour,nb_buts_table
31,M0032,E16,3,4
643,M0644,E16,1,2
743,M0744,E06,5,6
777,M0778,E12,1,2
1345,M0206,E14,1,2
1564,M0425,E12,1,2


In [38]:
# (b) pour chaque joueur et chaque saison : buts (joueurs) == nb de buts marqués dans buts
nb_marques = buts_c.groupby(["saison", "buteur_id"]).size().rename("buts_table").reset_index().rename(columns={"buteur_id": "joueur_id"})
ctrl_j = joueurs_c.merge(nb_marques, on=["saison", "joueur_id"], how="left")
ctrl_j["buts_table"] = ctrl_j["buts_table"].fillna(0).astype(int)
assert (ctrl_j["buts"] == ctrl_j["buts_table"]).all()
print("Joueurs-saisons incohérents :", (ctrl_j["buts"] != ctrl_j["buts_table"]).sum())

Joueurs-saisons incohérents : 0


**Interprétation 4.8 :** après nettoyage, tous les scores des matchs correspondent exactement au nombre de buts enregistrés et le total de buts de chaque joueur correspond à la table `buts` : les trois fichiers sont cohérents entre eux. Avec le fichier brut, 6 scores ne correspondent plus : ce sont les 6 buts saisis en double, qui gonflent le nombre de lignes de but de l'équipe concernée par rapport au score réel.

# Partie 5 — Export

Installation du module openpyxl

In [39]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
classements = classement.copy()
with pd.ExcelWriter("C:/Users/admin/Downloads/tp2_football_data/data/rapport_ligue.xlsx") as writer:
    classements.to_excel(writer, sheet_name="Classements", index=False)
    buteurs.to_excel(writer, sheet_name="Buteurs_2025-2026", index=False)
    affluence.to_excel(writer, sheet_name="Affluence_2025-2026", index=False)
print("rapport_ligue.xlsx exporté :", pd.ExcelFile("C:/Users/admin/Downloads/tp2_football_data/data/rapport_ligue.xlsx").sheet_names)

rapport_ligue.xlsx exporté : ['Classements', 'Buteurs_2025-2026', 'Affluence_2025-2026']


**Interprétation 5.1 :** le fichier `rapport_ligue.xlsx` contient les trois feuilles demandées (Classements, Buteurs_2025-2026, Affluence_2025-2026).

# Bonus — La course au titre 2025-2026

**B.1** Points cumulés de chaque club après chaque journée.

In [41]:
course = resultats[resultats["saison"] == "2025-2026"].sort_values(["equipe_id", "journee"]).copy()
course["pts_cumules"] = course.groupby("equipe_id")["points"].cumsum()
course = course.merge(equipes_c[["equipe_id", "nom"]], on="equipe_id")
course[["journee", "nom", "points", "pts_cumules"]].head()

,journee,nom,points,pts_cumules
0,1,Étoile du Cap-Vert,3,3
1,2,Étoile du Cap-Vert,3,6
2,3,Étoile du Cap-Vert,3,9
3,4,Étoile du Cap-Vert,3,12
4,5,Étoile du Cap-Vert,1,13


**B.2** Tableau journée × club et leader après chaque journée.

In [42]:
course_titre = course.pivot_table(index="journee", columns="nom", values="pts_cumules")   # à garder pour la séance 3

# Leader après chaque journée, départagé comme le règlement : points, puis différence de buts, puis buts marqués
course["diff_cumulee"] = (course["buts_pour"] - course["buts_contre"]).groupby(course["equipe_id"]).cumsum()
course["bp_cumules"] = course.groupby("equipe_id")["buts_pour"].cumsum()
leader = (
    course.sort_values(["journee", "pts_cumules", "diff_cumulee", "bp_cumules"], ascending=[True, False, False, False])
    .groupby("journee").head(1)
    .set_index("journee")["nom"]
)
nb_leaders = course_titre.eq(course_titre.max(axis=1), axis=0).sum(axis=1)
changements = (leader != leader.shift()).iloc[1:].sum()
print("Leader après chaque journée (extrait) :")
print(leader.iloc[[0, 4, 9, 14, 19, 24, 29, 34, 37]].to_string())
print("\nNombre de changements de leader :", changements)
print("Journées où la 1re place est partagée en points :", (nb_leaders > 1).sum())
print("\nSuite des leaders :", " > ".join(leader[leader != leader.shift()].tolist()))
course_titre.tail()

Leader après chaque journée (extrait) :
journee
1      Presqu'île Racing
5     Étoile du Cap-Vert
10    Étoile du Cap-Vert
15     Presqu'île Racing
20     Presqu'île Racing
25     Presqu'île Racing
30     Presqu'île Racing
35    Étoile du Cap-Vert
38     Presqu'île Racing

Nombre de changements de leader : 7
Journées où la 1re place est partagée en points : 7

Suite des leaders : Presqu'île Racing > Étoile du Cap-Vert > Sine Racing > Presqu'île Racing > Étoile du Cap-Vert > Presqu'île Racing > Étoile du Cap-Vert > Presqu'île Racing


nom,Banlieue Espoir,Baobab Sporting,Delta Club,Diourbel Sporting,Falaise AC,Ferlo United,Fleuve Athletic,Forêt FC,Kaffrine Athletic,Lagune Sportive,Niayes Académie,Océan Club,Palmiers FC,Presqu'île Racing,Sahel Olympique,Saloum United,Savane FC,Sine Racing,Sédhiou Olympique,Étoile du Cap-Vert
journee,,,,,,,,,,,,,,,,,,,,
34,56.0,62.0,45.0,40.0,33.0,40.0,41.0,41.0,28.0,38.0,54.0,45.0,57.0,66.0,33.0,48.0,37.0,54.0,31.0,68.0
35,59.0,63.0,45.0,40.0,36.0,43.0,44.0,44.0,28.0,41.0,55.0,48.0,57.0,66.0,33.0,49.0,38.0,54.0,32.0,69.0
36,62.0,66.0,48.0,43.0,36.0,46.0,47.0,47.0,28.0,41.0,55.0,48.0,60.0,69.0,33.0,49.0,39.0,54.0,33.0,69.0
37,63.0,67.0,48.0,44.0,37.0,47.0,48.0,47.0,28.0,42.0,58.0,49.0,61.0,72.0,36.0,50.0,40.0,55.0,34.0,70.0
38,66.0,67.0,51.0,45.0,40.0,47.0,48.0,47.0,28.0,42.0,59.0,50.0,64.0,75.0,39.0,53.0,40.0,56.0,34.0,73.0


**Interprétation B.2 :** la tête du classement a changé de mains **7 fois** en 2025-2026 (leaders successifs : Presqu'île Racing, Étoile du Cap-Vert, Sine Racing, Presqu'île Racing, Étoile du Cap-Vert, Presqu'île Racing, Étoile du Cap-Vert, puis Presqu'île Racing), et la première place a été partagée en points lors de 7 journées (départage par différence de buts puis buts marqués). La course s'est donc jouée jusqu'au bout entre Presqu'île Racing et Étoile du Cap-Vert, et c'est Presqu'île Racing qui prend le titre avec 2 points d'avance.

**B.3 :** le tableau `course_titre` est conservé pour la séance 3 (tracé de la course au titre).